# Agent Evaluation
Now that we've created an agent with tools, we'll evaluate it

In [0]:
%pip install -U -qqqq mlflow-skinny[databricks] langgraph==0.3.4 databricks-langchain databricks-agents uv
dbutils.library.restartPython()

In [0]:

from config import *

In [0]:
%load_ext autoreload
%autoreload 2

### 2.0 Create an MLflow Experiment

> Setting the experiment avoids MLflow using the default experiment name and keep things organized

In [0]:
import mlflow

# build a workspace path under your user home
user = spark.sql("select current_user()").first()[0]          # e.g. 'robert.leach@databricks.com'
exp_path = f"/Users/{user}/medtech_sc_agent_experiments"

# set the experiment (and end any stray run first)
mlflow.end_run()
mlflow.set_experiment(exp_path)

print("Experiment path:", exp_path)

#### 2.1 Unit Test the Agent, tracing the chain of thought of the agent

In [0]:
from supply_chain_agent import AGENT

# response = AGENT.predict({
#     "messages": [
#         {
#             "role": "user",
#             "content": "I heard the weather in New York is going to be hot tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me the results"
#         }
#     ]
# })

response = AGENT.predict({
    "messages": [
        {
            "role": "user",
            "content": "I heard the weather in New York is going to be hot tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me the results"
        }
    ]
})



display(response)

### 2.2 Log the agent in MLflow (locally) before proceeding with evals

> Setting the experiment avoids MLflow using the default experiment name and keep things organized

> See [MLflow - Models from Code](https://mlflow.org/docs/latest/models.html#models-from-code).

In [0]:
import datetime
import mlflow
from supply_chain_agent import tools, LLM_ENDPOINT_NAME
from databricks_langchain import VectorSearchRetrieverTool
from mlflow.models.resources import DatabricksFunction, DatabricksServingEndpoint
from unitycatalog.ai.langchain.toolkit import UnityCatalogTool

resources = [DatabricksServingEndpoint(endpoint_name=LLM_ENDPOINT_NAME)]
for tool in tools:
    if isinstance(tool, VectorSearchRetrieverTool):
        resources.extend(tool.resources)
    elif isinstance(tool, UnityCatalogTool):
        resources.append(DatabricksFunction(function_name=tool.uc_function_name))

input_example = {
    "messages": [
        {
            "role": "user",
            "content": "I heard the weather in Philadelphia is going to be hot tomorrow. Which in-transit shipments are at risk of temperature excursions, and what supplier escalation steps should I take? Also is there a backup supplier nearby? Email me the results"
        }
    ]
}


with mlflow.start_run(run_name=f"sc-agent-ut-{datetime.datetime.now():%Y%m%d}"):
    logged_agent_info = mlflow.pyfunc.log_model(
        name="medtech_sc_weather_aware_agent",      # under model artifacts, the top level folder            
        python_model="supply_chain_agent.py",       # agent python file 
        input_example=input_example,
        resources=resources,
        code_paths=["tools/custom_tools/"],      
        extra_pip_requirements=[
            "databricks-connect"
        ]
    )

### 2.3 Load model into a wrapper function to make running evals easier (simplier syntax)

> Load the model and create a prediction function so you can ask the model a question without the boilerplate message: role: user: conetent: syntax

In [0]:
# Load the model and create a prediction function so you can ask the model a question without the boilerplate message: role: user: conetent: syntax
logged_model_uri = f"runs:/{logged_agent_info.run_id}/medtech_sc_weather_aware_agent"
loaded_model = mlflow.pyfunc.load_model(logged_model_uri)

def predict_wrapper(query):
    # Format for chat-style models
    model_input = {
        "messages": [{"role": "user", "content": query}]
    }
    response = loaded_model.predict(model_input)
    
    messages = response['messages']
    return messages[-1]['content']

### 2.4 Load eval data


In [0]:
import pandas as pd

cases = [
    # Zimmer Biotech (ZB-SUP-2041)
    {
        "request": "According to Zimmer Biotech SOP ZB-SUP-2041, outline Tier-1, Tier-2, and Tier-3 escalation contacts and the SLA.",
        "expected_facts": [
            "Zimmer Biotech SOP ZB-SUP-2041 defines tiers",
            "Tier 1: logistics@zimmerbio.com",
            "Tier 2: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier 3: vp.ops@zimmerbio.com | +1-312-555-0100",
            "SLA: Acknowledge 1h; CAPA 24h"
        ],
    },
    {
        "request": "Per ZB-SUP-2041, when do we escalate to Tier-2 at Zimmer Biotech?",
        "expected_facts": [
            "Tier-2 trigger: temp > 8°F over max or delay > 24h",
            "Contacts: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier-1 is logistics@zimmerbio.com; Tier-3 is vp.ops@zimmerbio.com",
            "SLA: Acknowledge 1h; CAPA 24h"
        ],
    },

    # MedPro Devices (MP-SLA-007)
    {
        "request": "Summarize MedPro Devices SLA MP-SLA-007 and the escalation path.",
        "expected_facts": [
            "MedPro MP-SLA-007 requires on-time ≥ 98%",
            "Zero tolerance outside 2–8°C",
            "Escalation: shipping@medpro.com → operations@medpro.com → ceo.office@medpro.com",
            "If weather delay > 12h, reroute per SLA"
        ],
    },

    # Ortho Dynamics (OD-QA-115)
    {
        "request": "For Ortho Dynamics OD-QA-115, list L1/L2/L3 escalation contacts and time targets.",
        "expected_facts": [
            "Ortho Dynamics OD-QA-115 defines L1/L2/L3",
            "L1: warehouse.alerts@orthodynamics.com (≤2h)",
            "L2: dist.ops@orthodynamics.com (≤4h)",
            "L3: cmo.office@orthodynamics.com (≤8h)"
        ],
    },
    {
        "request": "What does OD-QA-115 require when a temperature excursion is detected?",
        "expected_facts": [
            "Temp excursion → isolate lot",
            "Notify QA per OD-QA-115",
            "Follow L1/L2/L3 timing if required",
            "Document in incident log"
        ],
    },

    # CardioNova (CN-COLD-003)
    {
        "request": "List the CardioNova CN-COLD-003 escalation triggers and contact chain.",
        "expected_facts": [
            "CN-COLD-003 triggers: delay > 24h, ambient > 10°C, severe weather",
            "Chain: supply@cardionova.com → qa@cardionova.com → compliance@cardionova.com",
            "Record temperatures at handoff per storage guidance"
        ],
    },

    # ValveX Labs (VX-QMS-221)
    {
        "request": "According to ValveX Labs VX-QMS-221, classify excursions by severity and actions.",
        "expected_facts": [
            "Minor (<1°C) → log only",
            "Moderate (1–3°C) → notify QA",
            "Critical (>3°C) → isolate and escalate",
            "Contacts: qa.supervisor@valvex.com; compliance@valvex.com"
        ],
    },

    # OrthoCore (Supplier Playbook)
    {
        "request": "Summarize OrthoCore escalation rules from Supplier Playbook.",
        "expected_facts": [
            "Notify ops@orthocore.com initially",
            "If delay > 24h or temp > 8°C, escalate to qa@orthocore.com",
            "Storage: maintain 2–8°C and record at handoff"
        ],
    },

    # MedAxis (Escalation Matrix)
    {
        "request": "Provide MedAxis escalation matrix contacts (L1/L2/L3).",
        "expected_facts": [
            "MedAxis Escalation Matrix defines L1/L2/L3",
            "L1: shipping@medaxis.io",
            "L2: ops@medaxis.io",
            "L3: exec@medaxis.io"
        ],
    },

    # Cross-check product storage (common across rows)
    {
        "request": "What is the standard storage requirement mentioned for these products?",
        "expected_facts": [
            "Maintain 2–8°C (≈36–46°F)",
            "Record temperatures at handoff"
        ],
    },

    # Product-specific: Balloon Catheter, Ortho Dynamics
    {
        "request": "For the Balloon Catheter (CATH-B08) under Ortho Dynamics, what are the escalation contacts and timing?",
        "expected_facts": [
            "Product: Balloon Catheter (CATH-B08)",
            "Supplier: Ortho Dynamics (OD-QA-115)",
            "L1: warehouse.alerts@orthodynamics.com (≤2h)",
            "L2: dist.ops@orthodynamics.com (≤4h)",
            "L3: cmo.office@orthodynamics.com (≤8h)"
        ],
    },

    # Product-specific: Aortic Valve, ValveX Labs
    {
        "request": "For Aortic Valve (VALVE-E11) at ValveX Labs, how are excursions classified and who is contacted?",
        "expected_facts": [
            "Product: Aortic Valve (VALVE-E11)",
            "Supplier: ValveX Labs (VX-QMS-221)",
            "Minor <1°C log; 1–3°C notify QA; >3°C isolate & escalate",
            "qa.supervisor@valvex.com; compliance@valvex.com"
        ],
    },

    # Product-specific: Hip Implant Set at Zimmer
    {
        "request": "For Hip Implant Set (IMPL-A21) from Zimmer Biotech, list Tier-1/2/3 contacts and SLA.",
        "expected_facts": [
            "Product: Hip Implant Set (IMPL-A21) | Supplier: Zimmer Biotech",
            "Tier 1: logistics@zimmerbio.com",
            "Tier 2: qa@zimmerbio.com; director.supply@zimmerbio.com",
            "Tier 3: vp.ops@zimmerbio.com | +1-312-555-0100",
            "Acknowledge 1h; CAPA 24h"
        ],
    },

    # Product-specific: Cardio Stent Kit at MedPro
    {
        "request": "For Cardio Stent Kit (STENT-C13) at MedPro, summarize SLA and escalation ladder.",
        "expected_facts": [
            "Product: Cardio Stent Kit (STENT-C13) | Supplier: MedPro Devices",
            "On-time ≥ 98%; zero tolerance outside 2–8°C",
            "Escalation: shipping@medpro.com → operations@medpro.com → ceo.office@medpro.com",
            "Reroute if weather delay > 12h"
        ],
    },
]

eval_df = pd.DataFrame(cases)

# Format for mlflow.genai.evaluate (inputs + expected_response)
eval_data = [
    {
        "inputs": {"query": row["request"]},
        "expected_response": "\n".join(row["expected_facts"]),
    }
    for _, row in eval_df.iterrows()
]
print("✅ Wrote eval_dataset")

### 2.5 Define LLM Judges


In [0]:
from mlflow.genai.scorers import Guidelines, Safety
import mlflow.genai

scorers = [
    Guidelines(
        name="factual_keypoints",
        guidelines="""Pass if response covers most SOP facts (≈60%+):
        - Correct supplier and SOP reference
        - At least two valid contacts or tiers
        - Any correct SLA or timing rule
        - Correct storage range or escalation trigger
        Minor wording or format differences are fine.
        Fail only if mostly incorrect or missing core info."""
    ),
    Guidelines(
        name="jj_tone",
        guidelines="""Use Jackson & Jackson professional tone:
        - Clear summary + concise bullets
        - Professional, factual, and on-brand
        - Including SOP or supplier IDs is OK
        - No logs, JSON, or system text
        Fail only if casual, chatty, or off-brand."""
    ),
    Guidelines(
        name="actionability_min",
        guidelines="""Pass if includes at least two concrete, ordered steps tied to SOP
        (e.g., “Email L1”, “Escalate to L2 in 4h”). 
        Skip backup inventory or extra logistics.
        Fail only if vague or lacks clear actions."""
    ),
    Safety(
        name="regulatory_and_safety_compliance",
        criteria=[
            "No patient PII; supplier contact emails are OK.",
            "No verbatim SOP text or internal logs.",
            "Stay within approved SOP and compliance scope.",
            "Maintain professional supply-chain language.",
        ],
    ),
]

### 2.6 Run evals


In [0]:
print("Running evaluation...")
with mlflow.start_run(run_name=f"sc-agent-evals-{datetime.datetime.now():%Y%m%d}"):
    results = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=predict_wrapper, 
        scorers=scorers,
    )

#### 2.6 Interate
>Based on evals go back to the agent code (.py file) and change our prompt to be more specific, more guildines, reduce marketing fluff, be more actionable, etc.

### 2.7 After satisified with evals, log model to Unity Catalog


In [0]:
from databricks.sdk import WorkspaceClient
import os

uc_model_name = f"{WORKSHOP_CATALOG}.{WORKSHOP_SCHEMA}.medtech_sc_weather_aware_agent"

# register the model to UC
uc_registered_model_info = mlflow.register_model(model_uri=logged_agent_info.model_uri, name=uc_model_name)

In [0]:
from IPython.display import display, HTML

# Retrieve the Databricks host URL
workspace_url = spark.conf.get('spark.databricks.workspaceUrl')

# Create HTML link to created agent
html_link = f'<a href="https://{workspace_url}/explore/data/models/{WORKSHOP_CATALOG}/{WORKSHOP_SCHEMA}/medtech_sc_weather_aware_agent" target="_blank">Go to Unity Catalog to see Registered Agent</a>'
display(HTML(html_link))

### 2.8 Deploy to model serving (and spin up human feedback app)


In [0]:
from databricks import agents

# Deploy the model to the review app and a model serving endpoint


agents.deploy(
    uc_model_name,
    uc_registered_model_info.version,
    scale_to_zero=True,
    environment_vars={
        "WORKSHOP_CATALOG": WORKSHOP_CATALOG,
        "WORKSHOP_SCHEMA": WORKSHOP_SCHEMA,
        "USER_SCHEMA": WORKSHOP_SCHEMA,
    },
    tags={"endpointSource": "MedTech Weather Aware Agent"},
)